# Module 5.2: The Training Loop

Once we have our Cross-Entropy Loss (which tells us how wrong the model is), we need to actually update the millions of parameters inside the Transformer to be *less* wrong.

In this notebook, we look at the magic of **Backpropagation** and **Optimizers**, the engine that drives all Neural Network training.

## 1. Backpropagation (Calculus in Disguise)

PyTorch has an engine called `Autograd` that mathematically remembers every matrix multiplication, addition, and scaling function our data went through during the Forward Pass.

When we call `loss.backward()`, PyTorch walks backward across the entire graph and calculates the **Gradient** (the slope) for every single weight in our Transformer. A gradient tells us: *"If I increase this weight by a tiny amount, will the loss go up or down, and by how much?"*

One important detail: PyTorch **accumulates** (adds up) gradients into each weight's `.grad` every time you call `backward()`. It does not clear them automatically. That's why every training step starts with `optimizer.zero_grad()` — otherwise this step's gradients would pile on top of the last step's.

> **You already built this.** In **Module 1.3** you wrote this exact backward pass by hand in NumPy — and verified that autograd reproduces your numbers to the last decimal. `loss.backward()` is your seven-line `backward()` function, generalized to every operation PyTorch knows. If that module is fuzzy, revisit it: everything below assumes you know what the "magic" actually does.

## 2. The Optimizer (AdamW)

Subtracting the raw gradients from our weights is called basic Stochastic Gradient Descent (SGD). However, for deep LLMs, this causes training to be unstable.

Modern LLMs use **AdamW** (Adaptive Moment Estimation with Weight Decay). Adam tracks **two** running averages for every weight:
1. **First moment (momentum):** the running average of the gradient itself. If a weight keeps getting pushed in the same direction, this builds up speed in that direction.
2. **Second moment:** the running average of the *squared* gradient — i.e. how large that weight's gradients have typically been. Adam divides the update by (roughly) the square root of this, so each weight gets its own *effective* learning rate: weights with big, noisy gradients take smaller, calmer steps, and weights with tiny gradients get a relative boost.
3. **Weight Decay** (the "W"): gently pulls every weight slightly toward `0.0` each step. This prevents any single weight from growing huge (a form of regularization).

So Adam is not just momentum — it's momentum **plus** a per-weight scaling that comes from the second moment. Let's build a mini training loop!

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# 1. A tiny mock model (imagine this is our 96-layer Transformer!)
#    NOTE: real LLMs use a SwiGLU / SiLU activation in their feed-forward blocks,
#    not plain ReLU. We use ReLU here just to keep the demo simple.
model = nn.Sequential(
    nn.Linear(256, 512),
    nn.ReLU(),
    nn.Linear(512, 1000) # Outputting 1000 raw logits for 1000 vocab words
)

# 2. The Optimizer (AdamW)
# We hand it the actual memory addresses of our model's weights so it can edit them!
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

# 3. A dummy dataset 
x_inputs = torch.randn(32, 256)   # Batch of 32 contexts
y_targets = torch.randint(0, 1000, (32,)) # 32 target words to predict

# ----------------------------------------------------------------------------
# A quick peek at autograd: a weight's .grad is empty until we call backward().
first_weight = model[0].weight
print("Grad BEFORE backward():", first_weight.grad)            # -> None
demo_loss = nn.functional.cross_entropy(model(x_inputs), y_targets)
demo_loss.backward()
print("Grad AFTER  backward():", first_weight.grad.shape,
      "| a few values:", first_weight.grad.flatten()[:3].tolist())
optimizer.zero_grad()  # wipe that demo gradient before the real loop starts
print("-" * 60)
# ----------------------------------------------------------------------------

# IMPORTANT: we reuse the SAME fixed batch every epoch below. This demonstrates
# the MECHANICS of the loop, but the model is just MEMORIZING one batch — it is
# NOT learning to generalize. Real training streams fresh batches of new data.

# --- THE SACRED TRAINING LOOP ---
epochs = 5
for epoch in range(epochs):
    
    # Step A: Clear old gradients (they accumulate by default — see note above!)
    optimizer.zero_grad()
    
    # Step B: Forward Pass (Predict)
    logits = model(x_inputs)
    
    # Step C: Calculate Loss (How wrong were we?)
    loss = nn.functional.cross_entropy(logits, y_targets)
    
    # Step D: Backward Pass (Calculate the slopes using Calculus)
    loss.backward()
    
    # Step E: Optimize! (edit the weights using the gradients)
    optimizer.step()
    
    print(f"Epoch {epoch+1}/5 | Loss: {loss.item():.4f}")

print("Loss drops because the model is memorizing this one fixed batch.")

## 3. Learning-Rate Schedules (Warmup + Cosine)

So far the learning rate has been a single constant. Real LLM training almost never does that — it follows a **schedule** with two phases:

1. **Linear warmup** (the first few hundred/thousand steps): start near zero and ramp up. Why? Adam's running averages ($m$, $v$ — you built them in Module 1.3) are garbage estimates at step 1, and a full-size step on top of random weights can wreck the initialization before learning even starts.
2. **Cosine decay**: glide smoothly from the peak down to ~10% of it. Big steps early (cover ground), small steps late (settle precisely into the minimum instead of bouncing around it).

This warmup + cosine shape is what GPT-3, Llama, and virtually every modern LLM trained with. (Our capstone in Module 5.3 keeps a constant lr — at 800 steps on a tiny model, a schedule isn't worth the extra moving part. Now you know it's a *choice*, not an omission.)

In [ ]:
import math
import matplotlib.pyplot as plt

max_steps, warmup = 1000, 100
max_lr, min_lr = 3e-4, 3e-5

def lr_at(step):
    if step < warmup:                                       # phase 1: linear warmup
        return max_lr * (step + 1) / warmup
    progress = (step - warmup) / (max_steps - warmup)       # phase 2: cosine decay
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

plt.figure(figsize=(7, 3.5))
plt.plot([lr_at(s) for s in range(max_steps)])
plt.xlabel("training step"); plt.ylabel("learning rate")
plt.title("Warmup + cosine decay -- the standard LLM schedule")
plt.tight_layout(); plt.show()

# Wiring it into PyTorch: LambdaLR multiplies the optimizer's base lr by your
# function's *ratio*. You then call scheduler.step() right after optimizer.step().
demo_opt = torch.optim.AdamW(model.parameters(), lr=max_lr)
scheduler = torch.optim.lr_scheduler.LambdaLR(demo_opt, lambda s: lr_at(s) / max_lr)

print("What the optimizer actually uses:")
for step in range(5):
    # ... forward / backward would go here ...
    print(f"  step {step}: lr = {scheduler.get_last_lr()[0]:.2e}")
    demo_opt.step()        # weights update first (a no-op here -- no gradients)...
    scheduler.step()       # ...then the schedule advances
print(f"  ...")
print(f"  step 100 (peak): lr = {lr_at(100):.2e}")
print(f"  step 999 (end):  lr = {lr_at(999):.2e}")

## 4. What a Real (Big) Training Run Adds

Our five-step loop *is* the real loop — but a serious run wraps four practical layers around it. You've now built enough to understand each one in a paragraph:

- **Gradient clipping.** One bad batch can produce a huge gradient that catapults the weights into nonsense. `torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)` — inserted between `backward()` and `step()` — rescales any update whose overall norm exceeds the cap. (Module 5.4 shows an explosion, and this seatbelt, live.)
- **Gradient accumulation.** Want an effective batch of 1M tokens on a GPU that fits 32K? Run 32 micro-batches, calling `loss.backward()` each time *without* `zero_grad()` — the gradients **add up** (the same accumulation we normally zero out, used on purpose) — then `step()` once.
- **Mixed precision.** Store/compute in 16-bit (`torch.bfloat16`) where it's safe, keep a 32-bit master copy where it isn't: ~half the memory, much faster on modern GPUs. In code it's a context manager:
  ```python
  with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
      logits, loss, _ = model(x, y)
  ```
  (Same tradeoff as quantization, Module 9.1 — precision for memory — applied at *training* time.)
- **Parallelism.** One GPU is never enough. **Data parallel** (DDP): every GPU holds a full model copy, sees a different slice of the batch, and gradients are averaged before each step — the loop itself is unchanged. When the model doesn't *fit* on one GPU, **FSDP / tensor parallel** shard the weights themselves across devices. Module 8.2 (Scaling Laws) is the "why" behind all this hardware.

None of these change the mental model: forward, loss, backward, step. They're logistics.

## Summary

This small 5-step loop is the engine that trains everything from a small digit classifier (MNIST) up to a 400 Billion parameter Llama 3 model running on a cluster of 16,000 GPUs.

Once the loss hits its floor across trillions of web tokens, the result is a **Foundation Model** (or Base Model). It knows grammar, facts, and logic, but it doesn't know how to chat. If you say "Hello", it might complete it with "Hello, World!" instead of "Hi, how can I help you today?". 

To turn it into an assistant, we need **Module 6: Fine-Tuning and Alignment**!

### 🏋️ Try it yourself

The learning rate controls how big each weight update is. Too small and training crawls; too big and it explodes.

Try this:
1. Copy the training loop above but use plain **SGD** with a huge `lr=50`. Watch the loss **diverge** — it shoots up (or turns into `nan`) instead of shrinking.
2. Then try a sensible `lr` like `0.1` and confirm it behaves.
3. Bonus: print `first_weight.abs().max()` each epoch in the diverging run to watch the weights blow up.

(We switch to SGD here because its raw, unscaled steps make divergence obvious; AdamW's per-weight scaling can mask it on a tiny memorized batch.)

In [ ]:
# Your turn: watch a too-large learning rate blow training up.
# We use a small low-capacity model so it CAN'T just memorize the batch in one
# giant step -- that way an oversized learning rate genuinely overshoots.
torch.manual_seed(0)
small_x = torch.randn(64, 8)
small_y = torch.randint(0, 4, (64,))

diverge_model = nn.Linear(8, 4)
bad_optimizer = torch.optim.SGD(diverge_model.parameters(), lr=500.0)  # way too big!

for epoch in range(6):
    bad_optimizer.zero_grad()
    loss = nn.functional.cross_entropy(diverge_model(small_x), small_y)
    loss.backward()
    bad_optimizer.step()
    max_weight = diverge_model.weight.abs().max().item()
    print(f"Epoch {epoch+1}/6 | Loss: {loss.item():.4f} | largest weight: {max_weight:.2f}")

print("\nNotice the loss climbing (and weights ballooning) -- that's divergence. Try lr=0.1 to fix it.")